# Technical Interview Walkthrough: Retail Promo Forecasting

## 🚀 The Thesis
> "When data signal collapses due to feature saturation, model complexity becomes a liability, not an asset."

In this project, I diagnosed a structural failure in advanced Machine Learning models (LightGBM/Prophet) which were unable to outperform a simple Naive Persistence baseline on the Dominick's Finer Foods dataset. This walkthrough explains the **why** behind the failure and my **causal inference strategy** for measuring promotion lift.

## 1. Phase 0: Data Integrity Audit
**Pitch:** *"Raw retail data is noisy and suspect. I didn't just model it; I audited it."*

- **Filter:** Used the 'OK' quality flag to prune suspect store-weeks (outages, errors).
- **Integrity:** Implemented weighted-average pricing for multi-unit transactions.
- **Reproducibility:** Built a `--task prepare` pipeline to cache the 'clean truth' in `data/processed/`.

## 2. Phase 1: Causal Inference (Fixed Effects)
**Question:** *"How do you know it was the promotion that caused the sales lift?"*

**Strategy:** High-Dimensional Fixed Effects (HDFE).
- **Store FE:** Absorbs unobserved location baseline demand.
- **Time FE (Month):** Absorbs macro-seasonal trends.
- **Coefficients:** Isolated a stable ~1,900 unit lift, independent of store type or seasonality.

## 3. Phase 2: The Forecasting Showdown
**The Comparison:**
1. **Naive Baseline:** $y_t = y_{t-1}$
2. **Prophet:** Additive Bayesian seasonality.
3. **LightGBM:** Gradient Boosting with 12+ engineered temporal lags.

**The Result:** Naive wins.

## 4. Final Diagnosis: Feature Degeneracy
**The 'Aha' Moment:** The holdout period had **95% promotional intensity**.

**Conclusion:** Because the predictor (`Promo`) basically became a constant (1), the ML models lost their exogenous variance. In this 'Saturated Regime', the series is pure autocorrelation, making persistence (Naive) the mathematically optimal choice.

**Business Recommendation:** Don't waste compute on aggregate chain forecasting in saturated markets. Shift focus to **Store-Level Heterogeneity** where variance still exists.